# Data exploration of trace gasses dataset

## Data observation:
**non-standard file format**

`*.ag` is not standard file format (like `*.csv`). However, the files contain tab-separated values

**Data fragmentation and invalid files**
Data is fragmented in multiple files in multiple directories. Each valid file contains Timestamp, measured value and instrument health data. alongside files with measurement data, there are also log files.

**Inconsistent files structure** 
File header is in some cases comma-separated and in other cases tab separated

```
Time	CO(ppm)	p_sample(in-hg-a)	f_sample(ccm)	co_meas(mV)	co_ref(mV)	mr_ratio	t_bench(C)	t_wheel(C)	t_oven(C)	t_box(C)
```
vs.
```
Time,CO(ppm),p_sample(in-hg-a),f_sample(ccm),co_meas(mV),co_ref(mV),mr_ratio,t_bench(C),t_wheel(C),t_oven(C),t_box(C)
```

**Invalid timestamps**

Example on the 2nd line (`1.179`)
Example:
```
231129165004	1.179	48.0	62.0	46.0	40.8	0.404	25.5	1515.0	3267.9	2759.3	
1.179	25.5	1512.0	3267.8	2758.3	1.179	48.0	62.7	46.0	
231129165024	40.8	0.405	25.5	1514.0	3271.1	2761.3	1.179	48.0	61.8	46.0
```

**Corrupted files**

invalid characters ath the end of some files - manually corrected

Example files with corrupted content:
- `/Tvarminne/Trace_gases/CO/2025/CO_tele_250507.ag`
- `/Tvarminne/Trace_gases/CO/2025/CO_tele_250721.ag`

**Invalid recods** 

In file `/Tvarminne/Trace_gases/CO/2025/CO_tele_251110.ag`

```
251110050800	0.059	29.0	1687.0	4225.5	3555.9	1.18	47.9	62.0	46.0	40.8	
251110050820	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110050911	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051003	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051054	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
251110051146	nan	nan	nan	nan	nan	nan	nan	nan	nan	nan	
```

Many files miss value in many records.

**Inconsistency in column order**

SO2 has values in third column insted of second

```
Time		flag		SO2		hiso2		intt		rctt		pgast		pres		smplfl		pmtv		lmpv		lmpi
221221000047	4C000000	0.20	0.00	28.0	44.8	0.00	723.6	0.476	-714.1	1015	90.00*	
221221000147	4C000000	0.20	0.00	28.0	44.8	0.00	723.6	0.476	-714.1	1015	90.00*	
221221000247	4C000000	0.20	0.00	28.0	44.8	0.00	723.6	0.476	-714.1	1015	90.00*	
```

## Data preprocessing
To address challenges indentified in manula data observation, raw data is preprocessed and saved as single parquet file for each measure. Log data is ignored as well as raws with invalid data (i.e. timestamp not convertabl to datetime)

As a result, there are 4 parquet files:

- CO.parquet
- NO.parquet
- O3.parquet
- SO2.parquet

Each file contains only 2 rows, "Time" and measure name (i.e. "CO"). Other column present in raw data are ignored.

___

Import of libraries, defined constants:

In [49]:
import pandas as pd
from pathlib import Path

PREPROCESS = True

DATA_BASE_PATH = "../local/Tvarminne/Trace_gases"
DATA_GASSES = ["CO", "NO", "O3", "SO2"]

PREPROCESS_SUBPATH = "preprocessed"

PREPROCESSED_DATA_PATH = f"{DATA_BASE_PATH}/{PREPROCESS_SUBPATH}"

Data preprocessing code - combining data into single file for each gas:

In [50]:
def preprocess_data():
    for gas in DATA_GASSES:
        print(f"Preprocessing data for: {gas}...")
        path = Path(f"{DATA_BASE_PATH}/{gas}")
        files_to_process = path.glob("*/*.ag")
        dfs = []

        for file in files_to_process:
            # reading file, ignoring header
            df = pd.read_csv(file, skiprows=1, header=None, sep=r'\s+', engine='python')
            # keepng first 2 columns only (timestamp, value)
            if gas == "SO2":
                df = df.iloc[:, [0, 2]]
            else:
                df = df.iloc[:, [0, 1]]

            # setting column names
            df.columns = ["Time", gas]

            # Converting first column (timestamps) to numeric, coercing errors to NaN
            df["Time"] = pd.to_numeric(df["Time"], errors='coerce')
            df["Time"] = pd.to_datetime(df["Time"], format='%y%m%d%H%M%S', errors='coerce')
            df[gas] = pd.to_numeric(df[gas], errors='coerce')
            invalid_rows = df[df.isna().any(axis=1)]
            if len(invalid_rows) > 0:
                print(f"   Found {len(invalid_rows)} rows with invalid values in file {file}:")
            # Dropping rows with non-numeric Time values or missing values
            df = df.dropna()
            # setting Time as index
            df.set_index("Time", inplace=True)
            # appending to list of dataframes for this gas
            dfs.append(df)
        
        df = pd.concat(dfs, ignore_index=False)
        df = df.sort_index()
        new_path = Path(f"{PREPROCESSED_DATA_PATH}/{gas}.parquet")
        new_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(new_path, index=True)
        print(f"Saved data for {gas} to {new_path}")

Trigerring data preprocessing only if not yet preprocessed - identified by existance of directory with preprocessed data:

In [51]:
preprocessd_data_path = Path(PREPROCESSED_DATA_PATH)
if not preprocessd_data_path.exists():
    print(f"Preprocessed data not found at {PREPROCESSED_DATA_PATH}. Starting preprocessing...")
    preprocess_data()
else:
    print(f"Preprocessed data already exists at {PREPROCESSED_DATA_PATH}. Skipping preprocessing.")

Preprocessed data not found at ../local/Tvarminne/Trace_gases/preprocessed. Starting preprocessing...
Preprocessing data for: CO...
   Found 1679 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_251125.ag:
   Found 1678 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_251111.ag:
   Found 997 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250812.ag:
   Found 2 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250711.ag:
   Found 3 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250903.ag:
   Found 2 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250422.ag:
   Found 3 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250826.ag:
   Found 1677 rows with invalid values in file ../local/Tvarminne/Trace_gases/CO/2025/CO_tele_250725.ag:
   Found 2 rows with invalid values in fi